# Main microbial ANCOVA models using laboratory EC1:1

Reproduces Table 4.4. Total bacterial and fungal abundance are transformed with log(x + 1); richness is retained on its original scale. Type II tests use HC3 heteroscedasticity-consistent covariance estimates.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

SOIL_FILE = DATA_DIR / "soil_data.xlsx"
SOIL_SHEET = "PredictionNewSamplesSalinity_Fi"

if not SOIL_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {SOIL_FILE}\n"
        "Place the soil dataset in the data folder and name it 'soil_data.xlsx', "
        "or update SOIL_FILE in this notebook."
    )

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import variance_inflation_factor

OUT_DIR = OUTPUTS_DIR / "06_ancova_main_lab_ec"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(SOIL_FILE, sheet_name=SOIL_SHEET)
df.columns = df.columns.str.strip()

required_columns = [
    "SITE", "Layer", "EC (dS/m) 1:1", "ESP est",
    "TotalBacteria", "TotalFungi", "TYPES_BACTERIA", "TYPES_FUNGI",
    "SOC (%)", "Clay (%)", "pH"
]
missing = [column for column in required_columns if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["Zone"] = df["SITE"].astype(str).str.strip().str.upper()
df = df[df["Zone"].isin(["PERKERRA", "TANA"])].copy()

for column in required_columns[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.rename(columns={
    "EC (dS/m) 1:1": "EC_lab",
    "ESP est": "ESP",
    "TYPES_BACTERIA": "BacterialRichness",
    "TYPES_FUNGI": "FungalRichness",
    "SOC (%)": "SOC",
    "Clay (%)": "Clay"
})

df["Layer_cat"] = pd.Categorical(
    df["Layer"].astype("Int64").astype(str),
    categories=["1", "2", "3"],
    ordered=True
)
df["Zone"] = pd.Categorical(
    df["Zone"], categories=["PERKERRA", "TANA"], ordered=True
)

df["log_TotalBacteria"] = np.log1p(df["TotalBacteria"])
df["log_TotalFungi"] = np.log1p(df["TotalFungi"])

print("Dataset shape:", df.shape)
print(pd.crosstab(df["Zone"], df["Layer_cat"]))


In [ ]:
responses = {
    "log_TotalBacteria": "Total bacteria, log-transformed",
    "log_TotalFungi": "Total fungi, log-transformed",
    "BacterialRichness": "Bacterial richness",
    "FungalRichness": "Fungal richness",
}

formula_template = """
{response} ~ C(Layer_cat) * C(Zone)
+ EC_lab * C(Zone)
+ ESP * C(Zone)
+ SOC + Clay + pH
"""

term_labels = {
    "C(Layer_cat)": "Depth",
    "C(Zone)": "Irrigation scheme",
    "C(Layer_cat):C(Zone)": "Depth × irrigation scheme",
    "EC_lab": r"EC$_{1:1}$ lab",
    "EC_lab:C(Zone)": r"EC$_{1:1}$ lab × irrigation scheme",
    "ESP": "ESP",
    "ESP:C(Zone)": "ESP × irrigation scheme",
    "SOC": "SOC",
    "Clay": "Clay",
    "pH": "pH",
    "Residual": "Residual",
}

all_tables = []
all_metrics = []

for response, response_label in responses.items():
    columns = [
        response, "Layer_cat", "Zone", "EC_lab",
        "ESP", "SOC", "Clay", "pH"
    ]
    model_data = df[columns].dropna().copy()
    formula = formula_template.format(response=response)

    model = smf.ols(formula=formula, data=model_data).fit()
    table = anova_lm(model, typ=2, robust="hc3").reset_index()
    table = table.rename(columns={
        "index": "term",
        "df": "df",
        "F": "F_value",
        "PR(>F)": "p_value"
    })
    table["Response"] = response_label
    table["Term_label"] = table["term"].map(term_labels).fillna(table["term"])
    table["N"] = int(model.nobs)
    all_tables.append(table)

    all_metrics.append({
        "Response": response_label,
        "N": int(model.nobs),
        "R2": model.rsquared,
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
    })

ancova_hc3 = pd.concat(all_tables, ignore_index=True)
metrics = pd.DataFrame(all_metrics)

ancova_hc3.to_excel(OUT_DIR / "MAIN_LAB_EC_typeII_HC3_full.xlsx", index=False)
metrics.to_excel(OUT_DIR / "MAIN_LAB_EC_model_metrics.xlsx", index=False)

display(ancova_hc3)
display(metrics)


In [ ]:
# VIF for the continuous predictors used in the ANCOVA models
continuous_predictors = ["EC_lab", "ESP", "SOC", "Clay", "pH"]
vif_data = df[continuous_predictors].dropna().copy()
vif_data = (vif_data - vif_data.mean()) / vif_data.std(ddof=0)
X = sm.add_constant(vif_data)

vif_table = pd.DataFrame({
    "Variable": X.columns,
    "VIF": [
        variance_inflation_factor(X.values, index)
        for index in range(X.shape[1])
    ]
})
vif_table.to_excel(OUT_DIR / "MAIN_LAB_EC_VIF.xlsx", index=False)
display(vif_table)


In [ ]:
# Table 4.4 contains the statistically significant HC3 Type II terms.
publication_table = ancova_hc3[
    (ancova_hc3["p_value"] < 0.05) &
    (ancova_hc3["term"] != "Residual")
].copy()

response_order = [
    "Total bacteria, log-transformed",
    "Total fungi, log-transformed",
    "Bacterial richness",
    "Fungal richness",
]
term_order = [
    "Depth", "Irrigation scheme", "Depth × irrigation scheme",
    r"EC$_{1:1}$ lab", r"EC$_{1:1}$ lab × irrigation scheme",
    "ESP", "ESP × irrigation scheme", "SOC", "Clay", "pH"
]

publication_table["Response"] = pd.Categorical(
    publication_table["Response"], categories=response_order, ordered=True
)
publication_table["Term_label"] = pd.Categorical(
    publication_table["Term_label"], categories=term_order, ordered=True
)
publication_table = publication_table.sort_values(
    ["Response", "Term_label"]
)[["Response", "Term_label", "df", "F_value", "p_value"]]

publication_table.to_excel(
    OUT_DIR / "table_4_4_ancova_hc3_significant_terms.xlsx",
    index=False
)
display(publication_table)
